# DIMER Workshop: Comparing Closed-Set Object Detectors

**Profile:** `E2E`  
**Mode:** `WORKSHOP`  
**Notebook specification:** `2.1`  
**Status:** Candidate comparative carrier  
**Default tier:** `STANDARD`  
**Canonical runtime:** NVIDIA Tesla T4 or equivalent

This notebook compares closed-vocabulary object detectors on **one controlled downstream detection task**:

- **RT-DETR R50-VD** — transformer / set-prediction detector
- **YOLOX-S** — lightweight anchor-free dense detector
- **YOLOX-X** — large YOLOX scale experiment (`FULL` tier)

The experiment controls what can meaningfully be controlled:

> **same pixels → same boxes → same class vocabulary → same 36/12/12 split → same common evaluator**

while preserving what should remain architecture-specific:

> **preprocessing semantics → detection formulation → assignment algorithm → loss → optimizer → adaptation scope**

### Canonical question

> How do set-prediction and dense anchor-free detectors transfer differently to the same small closed-vocabulary detection problem, and what accuracy/compute tradeoffs emerge once the data and evaluation are controlled?

### Standalone behavior

The notebook does not:

- clone a DIMER repository;
- import or download DIMER repository source;
- call DIMER workers/services;
- require credentials;
- require an upload on the default path.

RT-DETR runs through the pinned `transformers` implementation.

YOLOX is source-bound to Megvii's immutable upstream commit `419778480ab6ec0590e5d3831b3afb3b46ab2aa3`. This **candidate** notebook retrieves those eight upstream Python modules directly from that commit, checks their SHA-256 digests, applies the same three dependency-removal edits documented by the live DIMER YOLOX carrier, and executes them locally.

**Release gate:** before promoting this notebook from candidate to release-grade, carry those eight pinned YOLOX modules directly inside notebook cells, matching the existing DIMER YOLOX standalone-carrier pattern. No model/evaluation semantics need to change.

All built-in results are **synthetic tutorial/sample-sanity evidence**, not evidence of real traffic-sign detection performance.

## 0. Learning goals

By the end of this notebook, participants should be able to:

1. distinguish image classification from object detection;
2. explain set-prediction detection versus dense objectness-based detection;
3. interpret IoU, AP, AP50, and AP75;
4. understand why raw detector scores are not calibrated probabilities;
5. adapt pretrained COCO detectors to a new three-class vocabulary;
6. explain why RT-DETR and YOLOX should keep different native training objectives;
7. preserve train/validation/test ownership;
8. compare localization quality rather than AP50 alone;
9. inspect size, density, class-confusion, and threshold effects;
10. export and fresh-reload adapted detector artifacts; and
11. understand when closed-set detection is the wrong tool and an open-vocabulary detector is needed.

## 1. Notebook controls

In [ ]:
# @title Notebook controls
WORKSHOP_TIER = "STANDARD"  # @param ["STANDARD", "FULL"]

USE_BYOD = False            # @param {type:"boolean"}
BYOD_ZIP_PATH = ""          # @param {type:"string"}

OUTPUT_DIR = "outputs"      # @param {type:"string"}

DATASET_IMAGES = 60
DATASET_SEED = 0
SPLIT_SEED = 42
NEW_DATA_SEED = 2026

RTDETR_EPOCHS = 3
RTDETR_BATCH_SIZE = 4
RTDETR_LEARNING_RATE = 1e-4

YOLOX_EPOCHS = 6
YOLOX_BATCH_SIZE = 2
YOLOX_LEARNING_RATE = 1e-3

EVAL_SCORE_THRESHOLD = 0.01
YOLOX_EVAL_NMS = 0.65
DISPLAY_THRESHOLD = 0.30
MAX_EVAL_DETECTIONS = 100
IOU_THRESHOLDS = tuple(round(0.50 + 0.05*i, 2) for i in range(10))
CLASS_NAMES = ("stop-sign", "yield-sign", "speed-limit-sign")

if WORKSHOP_TIER not in {"STANDARD", "FULL"}:
    raise ValueError("WORKSHOP_TIER must be STANDARD or FULL")
if DATASET_IMAGES != 60:
    raise ValueError("Canonical fixture is pinned to 60 images")
if not 0 <= EVAL_SCORE_THRESHOLD <= 1:
    raise ValueError("EVAL_SCORE_THRESHOLD outside [0,1]")

MODEL_KEYS = ["rtdetr", "yolox_s"] + (["yolox_x"] if WORKSHOP_TIER == "FULL" else [])

from pathlib import Path
OUTPUT_ROOT = Path(OUTPUT_DIR)
WORK_ROOT = Path("work")

for path in [
    OUTPUT_ROOT / "data",
    OUTPUT_ROOT / "pre_adaptation",
    OUTPUT_ROOT / "adaptation",
    OUTPUT_ROOT / "validation",
    OUTPUT_ROOT / "frozen",
    OUTPUT_ROOT / "test",
    OUTPUT_ROOT / "artifacts",
    OUTPUT_ROOT / "figures",
    OUTPUT_ROOT / "new_data",
    OUTPUT_ROOT / "provenance",
    WORK_ROOT / "rtdetr_weights",
    WORK_ROOT / "yolox_weights",
    WORK_ROOT / "yolox_upstream",
    WORK_ROOT / "byod",
]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "tier": WORKSHOP_TIER,
    "models": MODEL_KEYS,
    "dataset_images": DATASET_IMAGES,
    "classes": CLASS_NAMES,
    "split_seed": SPLIT_SEED,
})

## 2. Install one common runtime

RT-DETR needs Transformers.

YOLOX uses only PyTorch, torchvision, NumPy, and Pillow once its pinned upstream model modules have been staged.

The package union remains small enough for a single Python 3.12 environment.

In [ ]:
# @title Install pinned runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "transformers==4.57.6",
    "safetensors==0.8.0",
    "numpy==2.5.3",
    "pillow==11.3.0",
    "huggingface-hub==0.36.2",
    "matplotlib>=3.9,<3.11",
    "pandas>=2.2,<3.1",
]

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"

if not SKIP_INSTALL:
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PINS],
        check=False, text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"pip install failed with exit code {completed.returncode}.\n"
            f"--- stderr (tail) ---\n{completed.stderr[-4000:]}\n"
            f"--- stdout (tail) ---\n{completed.stdout[-2000:]}"
        )
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageOps

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": importlib.metadata.version("transformers"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

# 3. Model identities

## RT-DETR R50-VD

- Model: `PekingU/rtdetr_r50vd`
- Revision: `df939e661d8c52e80608d1ec566561aabd25a4e7`
- License: Apache-2.0
- Approx. parameters: 42.7M
- Architecture: ResNet-50-vd → hybrid encoder → 6-layer deformable transformer decoder → 300 learned object queries
- Original vocabulary: 80 COCO classes
- Preprocessing: RT-DETR's pinned 640×640 processor

Pinned SafeTensors:

`model.safetensors`  
172,175,856 bytes  
SHA-256 `5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64`

## YOLOX-S

- Release: Megvii YOLOX `0.1.1rc0`
- Upstream code revision: `419778480ab6ec0590e5d3831b3afb3b46ab2aa3`
- License: Apache-2.0
- Parameters: 8,968,255
- 640×640 anchor-free candidate grid: 8,400 locations

Checkpoint:

`yolox_s.pth`  
72,089,125 bytes  
SHA-256 `f55ded7181e1b0c13285c56e7790b8f0e8f8db590fe4edb37f0b7f345c913a30`

## YOLOX-X (`FULL` only)

- Parameters: 99,071,455
- Depth 1.33 / width 1.25

Checkpoint:

`yolox_x.pth`  
793,463,373 bytes  
SHA-256 `5652330b6ae860043f091b8f550a60c10e1129f416edfdb65c259be6caf355cf`

In [ ]:
# @title Immutable model specifications
RTDETR_SPEC = {
    "model_id": "PekingU/rtdetr_r50vd",
    "revision": "df939e661d8c52e80608d1ec566561aabd25a4e7",
    "files": {
        "README.md": (9053, "4a0c10ddd0dbf6a2c6815cfc1613a5d74f3b0e7c8c77f0252212d3e5365e98cb"),
        "config.json": (5113, "2ed2a305c51eef46715eb755a02b2a266ecfb752936cc9574bb5714601c2742d"),
        "model.safetensors": (172175856, "5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64"),
        "preprocessor_config.json": (841, "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a"),
    },
}

YOLOX_VARIANTS = {
    "yolox_s": {
        "display_name": "YOLOX-S",
        "depth": 0.33,
        "width": 0.50,
        "parameters_reference": 8_968_255,
        "checkpoint": "yolox_s.pth",
        "bytes": 72_089_125,
        "sha256": "f55ded7181e1b0c13285c56e7790b8f0e8f8db590fe4edb37f0b7f345c913a30",
        "url": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth",
    },
    "yolox_x": {
        "display_name": "YOLOX-X",
        "depth": 1.33,
        "width": 1.25,
        "parameters_reference": 99_071_455,
        "checkpoint": "yolox_x.pth",
        "bytes": 793_463_373,
        "sha256": "5652330b6ae860043f091b8f550a60c10e1129f416edfdb65c259be6caf355cf",
        "url": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_x.pth",
    },
}

# 4. Build one deterministic 60-image closed-set detection task

This notebook uses its own synthetic traffic-sign generator derived from the existing DIMER RT-DETR and YOLOX fixtures.

Target vocabulary:

- `stop-sign`
- `yield-sign`
- `speed-limit-sign`

All images are exactly **640×640**.

That is important:

- RT-DETR normally squashes arbitrary inputs to 640×640;
- YOLOX normally preserves aspect ratio and letterboxes to 640×640.

For this canonical fixture, both see the same 640×640 source geometry, removing a major preprocessing confound.

In [ ]:
# @title Deterministic sign generator
import math
import hashlib
import io
import json

SCENE_SIZE = (640, 640)
MAX_OBJECTS = 3

def _font(size=22):
    try:
        return ImageFont.truetype("DejaVuSans-Bold.ttf", size)
    except Exception:
        return ImageFont.load_default()

def _stop_sign(draw, cx, cy, r):
    pts = []
    for i in range(8):
        angle = math.radians(22.5 + 45*i)
        pts.append((cx + r*math.cos(angle), cy + r*math.sin(angle)))
    draw.polygon(pts, fill=(195, 28, 35), outline="white")
    draw.line(pts + [pts[0]], fill="white", width=5)
    font = _font(max(12, int(r*0.34)))
    text = "STOP"
    bbox = draw.textbbox((0,0), text, font=font)
    tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
    draw.text((cx-tw/2, cy-th/2), text, fill="white", font=font)
    xs, ys = zip(*pts)
    return [min(xs), min(ys), max(xs), max(ys)]

def _yield_sign(draw, cx, cy, r):
    pts = [
        (cx, cy-r),
        (cx-r*0.92, cy+r*0.72),
        (cx+r*0.92, cy+r*0.72),
    ]
    draw.polygon(pts, fill="white", outline=(205,30,35))
    draw.line(pts + [pts[0]], fill=(205,30,35), width=8)
    xs, ys = zip(*pts)
    return [min(xs), min(ys), max(xs), max(ys)]

def _speed_limit_sign(draw, cx, cy, r, limit):
    w, h = r*1.28, r*1.55
    box = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
    draw.rounded_rectangle(box, radius=max(5,int(r*0.08)), fill="white", outline="black", width=5)
    font1 = _font(max(10,int(r*0.22)))
    font2 = _font(max(18,int(r*0.45)))
    title = "SPEED"
    tb = draw.textbbox((0,0), title, font=font1)
    draw.text((cx-(tb[2]-tb[0])/2, cy-h*0.33), title, fill="black", font=font1)
    text = str(limit)
    nb = draw.textbbox((0,0), text, font=font2)
    draw.text((cx-(nb[2]-nb[0])/2, cy-(nb[3]-nb[1])/2+5), text, fill="black", font=font2)
    return box

def generate_sign_dataset(n_images=60, seed=0, max_objects=3):
    if not 1 <= n_images <= 500:
        raise ValueError("n_images outside 1..500")
    rng = np.random.default_rng(seed)
    width, height = SCENE_SIZE
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records = []

    for image_index in range(n_images):
        image = Image.new("RGB", SCENE_SIZE, (232,236,240))
        draw = ImageDraw.Draw(image)
        tint = rng.integers(200,245,3)
        draw.rectangle([0,0,width,height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0,int(height*0.72),width,height], fill=(118,122,126))

        n_objects = int(rng.integers(1,max_objects+1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes, labels = [], []

        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28,29))
            cy += float(rng.integers(-28,29))
            radius = float(rng.integers(46,71))
            cx = min(max(cx, radius+6), width-radius-6)
            cy = min(max(cy, radius+6), height-radius-100)

            kind = CLASS_NAMES[int(rng.integers(0,len(CLASS_NAMES)))]
            draw.rectangle(
                [cx-5, cy+radius*0.6, cx+5, min(height-1, cy+radius+90)],
                fill=(112,112,116),
            )

            if kind == "stop-sign":
                box = _stop_sign(draw,cx,cy,radius)
            elif kind == "yield-sign":
                box = _yield_sign(draw,cx,cy,radius)
            else:
                box = _speed_limit_sign(draw,cx,cy,radius,int(rng.choice([30,50,60,80])))

            boxes.append([
                float(max(0,box[0])), float(max(0,box[1])),
                float(min(width,box[2])), float(min(height,box[3])),
            ])
            labels.append(kind)

        buf = io.BytesIO()
        image.save(buf, format="PNG")
        pixel_sha = hashlib.sha256(buf.getvalue()).hexdigest()
        ann_sha = hashlib.sha256(
            json.dumps({"boxes":boxes,"labels":labels}, separators=(",",":")).encode()
        ).hexdigest()

        records.append({
            "id": f"sign-{image_index:03d}",
            "image": image,
            "boxes": boxes,
            "labels": labels,
            "pixel_sha256": pixel_sha,
            "annotation_sha256": ann_sha,
        })

    return records

all_records = generate_sign_dataset(DATASET_IMAGES, DATASET_SEED, MAX_OBJECTS)

print({
    "images": len(all_records),
    "boxes": sum(len(r["boxes"]) for r in all_records),
    "class_counts": {
        name: sum(label==name for r in all_records for label in r["labels"])
        for name in CLASS_NAMES
    },
})

# 5. Deterministic 36 / 12 / 12 split and validation

No test image is used for training or model selection.

The notebook additionally checks:

- box geometry;
- image dimensions;
- class membership;
- class coverage in all three splits;
- no duplicate image bytes across splits.

In [ ]:
# @title Split and validate records
def split_records(records, seed=42):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(records))
    train_ids = set(order[:36].tolist())
    val_ids = set(order[36:48].tolist())
    test_ids = set(order[48:60].tolist())

    result = {"train":[], "validation":[], "test":[]}
    for i, record in enumerate(records):
        split = "train" if i in train_ids else "validation" if i in val_ids else "test"
        result[split].append({**record, "split":split})
    return result

def validate_records(records, class_names):
    seen_ids=set()
    seen_pixels=set()
    class_counts={name:0 for name in class_names}

    for r in records:
        if r["id"] in seen_ids:
            raise ValueError(f"duplicate id {r['id']}")
        seen_ids.add(r["id"])
        if r["pixel_sha256"] in seen_pixels:
            raise ValueError(f"duplicate image bytes at {r['id']}")
        seen_pixels.add(r["pixel_sha256"])

        if r["image"].size != SCENE_SIZE:
            raise ValueError(f"{r['id']}: canonical sample must be 640x640")
        if len(r["boxes"]) != len(r["labels"]) or not r["boxes"]:
            raise ValueError(f"{r['id']}: boxes/labels invalid")

        for box,label in zip(r["boxes"],r["labels"]):
            if label not in class_names:
                raise ValueError(f"{r['id']}: unknown label {label}")
            if len(box)!=4 or not np.isfinite(box).all():
                raise ValueError(f"{r['id']}: invalid box {box}")
            x0,y0,x1,y1=map(float,box)
            if not (0<=x0<x1<=640 and 0<=y0<y1<=640):
                raise ValueError(f"{r['id']}: out-of-bounds box {box}")
            class_counts[label]+=1
    return class_counts

splits = split_records(all_records, SPLIT_SEED)
for split_name, rows in splits.items():
    counts = validate_records(rows, CLASS_NAMES)
    missing=[k for k,v in counts.items() if v==0]
    if missing:
        raise RuntimeError(f"canonical split {split_name} lacks classes {missing}")
    print(split_name, len(rows), counts)

# Cross-split pixel leak check.
digest_to_split={}
for split_name, rows in splits.items():
    for r in rows:
        prior=digest_to_split.get(r["pixel_sha256"])
        if prior and prior!=split_name:
            raise RuntimeError(f"pixel content leaks across {prior}/{split_name}")
        digest_to_split[r["pixel_sha256"]]=split_name

train_records=splits["train"]
validation_records=splits["validation"]
test_records=splits["test"]

dataset_manifest = {
    "generator": "dimer-closed-set-signs-v1",
    "generator_seed": DATASET_SEED,
    "split_seed": SPLIT_SEED,
    "image_size": list(SCENE_SIZE),
    "class_names": list(CLASS_NAMES),
    "n_images": len(all_records),
    "n_boxes": sum(len(r["boxes"]) for r in all_records),
    "splits": {
        name: {
            "image_ids":[r["id"] for r in rows],
            "n_images":len(rows),
            "n_boxes":sum(len(r["boxes"]) for r in rows),
        }
        for name,rows in splits.items()
    },
    "images": [
        {
            "id":r["id"],
            "pixel_sha256":r["pixel_sha256"],
            "annotation_sha256":r["annotation_sha256"],
            "split":r["split"],
        }
        for rows in splits.values() for r in rows
    ],
}
dataset_manifest["dataset_sha256"] = hashlib.sha256(
    json.dumps(dataset_manifest["images"],sort_keys=True,separators=(",",":")).encode()
).hexdigest()

(OUTPUT_ROOT/"data"/"dataset_manifest.json").write_text(
    json.dumps(dataset_manifest,indent=2),encoding="utf-8"
)

print("dataset_sha256",dataset_manifest["dataset_sha256"])

In [ ]:
# @title Dataset gallery
fig, axes = plt.subplots(2,3,figsize=(12,8))
for ax,record in zip(axes.ravel(),train_records[:6]):
    ax.imshow(record["image"])
    for box,label in zip(record["boxes"],record["labels"]):
        x0,y0,x1,y1=box
        rect=plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=2)
        ax.add_patch(rect)
        ax.text(x0,y0-4,label,fontsize=8)
    ax.set_title(record["id"])
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/"figures"/"dataset_gallery.png",dpi=150,bbox_inches="tight")
plt.show()

# 6. Common detector evaluator

The evaluator is notebook-owned and receives only normalized predictions:

```text
[
  {"box":[x0,y0,x1,y1], "label":"stop-sign", "score":0.73},
  ...
]
```

It implements class-aware one-to-one matching, 101-point interpolated AP, and IoU thresholds 0.50:0.05:0.95.

These are **COCO-style tutorial metrics**, not `pycocotools` reproduction: there are no crowd regions, area ranges, or official COCO-specific rules.

In [ ]:
# @title IoU, AP, precision/recall helpers
def box_iou(a,b):
    ax0,ay0,ax1,ay1=map(float,a)
    bx0,by0,bx1,by1=map(float,b)
    ix0,iy0=max(ax0,bx0),max(ay0,by0)
    ix1,iy1=min(ax1,bx1),min(ay1,by1)
    iw,ih=max(0.0,ix1-ix0),max(0.0,iy1-iy0)
    inter=iw*ih
    area_a=max(0.0,ax1-ax0)*max(0.0,ay1-ay0)
    area_b=max(0.0,bx1-bx0)*max(0.0,by1-by0)
    union=area_a+area_b-inter
    return inter/union if union>0 else 0.0

def average_precision(predictions,references,class_names,iou_thresholds=IOU_THRESHOLDS):
    recall_points=np.linspace(0,1,101)
    per_threshold={}
    for threshold in iou_thresholds:
        per_class={}
        for name in class_names:
            scored=[]
            n_refs=0
            for dets,ref in zip(predictions,references):
                ref_boxes=[
                    box for box,label in zip(ref["boxes"],ref["labels"])
                    if label==name
                ]
                n_refs+=len(ref_boxes)
                claimed=[False]*len(ref_boxes)
                candidates=sorted(
                    (d for d in dets if d["label"]==name),
                    key=lambda d:-float(d["score"])
                )
                for det in candidates:
                    best,best_iou=-1,0.0
                    for j,ref_box in enumerate(ref_boxes):
                        if claimed[j]: continue
                        value=box_iou(det["box"],ref_box)
                        if value>best_iou:
                            best,best_iou=j,value
                    hit=best>=0 and best_iou>=threshold
                    if hit: claimed[best]=True
                    scored.append((float(det["score"]),hit))
            if n_refs==0:
                continue
            scored.sort(key=lambda x:-x[0])
            if not scored:
                per_class[name]=0.0
                continue
            tp=np.cumsum([1 if hit else 0 for _,hit in scored])
            fp=np.cumsum([0 if hit else 1 for _,hit in scored])
            recall=tp/n_refs
            precision=tp/np.maximum(tp+fp,1)
            precision=np.maximum.accumulate(precision[::-1])[::-1]
            sampled=np.zeros_like(recall_points)
            indices=np.searchsorted(recall,recall_points,side="left")
            valid=indices<len(precision)
            sampled[valid]=precision[indices[valid]]
            per_class[name]=float(sampled.mean())
        per_threshold[float(threshold)]=per_class

    means={
        t:(float(np.mean(list(v.values()))) if v else 0.0)
        for t,v in per_threshold.items()
    }
    return {
        "ap":float(np.mean(list(means.values()))) if means else 0.0,
        "ap50":means.get(0.5,0.0),
        "ap75":means.get(0.75,0.0),
        "per_class_ap50":per_threshold.get(0.5,{}),
        "n_images":len(references),
        "n_references":sum(len(r["boxes"]) for r in references),
    }

def operating_metrics(predictions,references,threshold=0.30,iou=0.50):
    tp=fp=fn=0
    matched_ious=[]
    for dets,ref in zip(predictions,references):
        filtered=[d for d in dets if float(d["score"])>=threshold]
        claimed=[False]*len(ref["boxes"])
        for det in sorted(filtered,key=lambda d:-float(d["score"])):
            best,best_iou=-1,0.0
            for j,(box,label) in enumerate(zip(ref["boxes"],ref["labels"])):
                if claimed[j] or label!=det["label"]: continue
                value=box_iou(det["box"],box)
                if value>best_iou:
                    best,best_iou=j,value
            if best>=0 and best_iou>=iou:
                claimed[best]=True
                tp+=1
                matched_ious.append(best_iou)
            else:
                fp+=1
        fn+=sum(not x for x in claimed)
    return {
        "precision50":tp/(tp+fp) if tp+fp else 0.0,
        "recall50":tp/(tp+fn) if tp+fn else 0.0,
        "mean_matched_iou":float(np.mean(matched_ious)) if matched_ious else 0.0,
        "false_positives_per_image":fp/len(references),
        "missed_objects_per_image":fn/len(references),
    }

empty_predictions=[[] for _ in validation_records]
empty_metrics=average_precision(empty_predictions,validation_records,CLASS_NAMES)
print({"empty_detector":empty_metrics})
assert empty_metrics["ap"]==0.0 and empty_metrics["ap50"]==0.0

# 7. Stage the pinned RT-DETR SafeTensors snapshot

All four files are acquired from the immutable Hugging Face revision and verified before model loading.

In [ ]:
# @title RT-DETR snapshot staging
from huggingface_hub import hf_hub_download

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as fh:
        for chunk in iter(lambda:fh.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

RTDETR_ROOT=WORK_ROOT/"rtdetr_weights"
for name,(size,digest) in RTDETR_SPEC["files"].items():
    path=Path(hf_hub_download(
        RTDETR_SPEC["model_id"],
        name,
        revision=RTDETR_SPEC["revision"],
        local_dir=str(RTDETR_ROOT),
    ))
    if path.stat().st_size!=size or sha256_file(path)!=digest:
        raise ValueError(f"RT-DETR asset integrity mismatch: {name}")

print("RT-DETR snapshot verified:",RTDETR_ROOT)

# 8. Stage source-bound upstream YOLOX modules

The official `yolox` Python package is intentionally not installed because its historical dependency pins are unsuitable for this pinned runtime.

Instead this candidate notebook fetches only the exact upstream files required for the model, pinned to:

`419778480ab6ec0590e5d3831b3afb3b46ab2aa3`

Each source file is SHA-256 checked before use.

Three declared compatibility edits match the live DIMER YOLOX carrier:

1. replace `loguru` with Python `warnings`;
2. route `bboxes_iou` / `meshgrid` to the local minimal ops module;
3. replace the OOM logger call with `warnings.warn`.

**Release gate:** these verified upstream modules should be carried directly into notebook cells before promotion.

In [ ]:
# @title Stage verified upstream YOLOX source
import urllib.request
import sys

YOLOX_CODE_REVISION="419778480ab6ec0590e5d3831b3afb3b46ab2aa3"
RAW_BASE=f"https://raw.githubusercontent.com/Megvii-BaseDetection/YOLOX/{YOLOX_CODE_REVISION}/"

UPSTREAM_FILES={
    "network_blocks.py":("yolox/models/network_blocks.py",6092,"f490240a674799f1b548ac4ca571c7f3e59d4e75efee8ebeb1e24e6e4e10d2a4"),
    "darknet.py":("yolox/models/darknet.py",6019,"368c9db45166e863691cf7460b314d047928f90e8744c492a673b6eff9b809f3"),
    "yolo_pafpn.py":("yolox/models/yolo_pafpn.py",3530,"8571af557f6caacda61e54813be4be2d73c4a0889e1f876ec5586057eb2eebde"),
    "losses.py":("yolox/models/losses.py",1677,"7fa685fb4653bab69d5880c3f749edeb8a4cabfe71fadf0acc5151c2eaa75702"),
    "yolo_head.py":("yolox/models/yolo_head.py",23339,"9fcdcf859b91c587d11926c0a8e3e423abdaa53504eb8ad2a17379ee405378bd"),
    "yolox.py":("yolox/models/yolox.py",1364,"3b88f3b19e27232ab5442b4d2c53b20d3f1e40f6f093871d6742e694368f6da0"),
    "coco_classes.py":("yolox/data/datasets/coco_classes.py",1296,"b38193c481a73f1f674cedab9e551b15b39b1a7aaed3e09e16505362cc54ad51"),
}
OPS_FILES=[
    ("yolox/utils/compat.py",310,"a8f0dec9e0566cf0a09a211bc08ce47bc7abd6dd75d239ee85d8c0c5fe3e07ce"),
    ("yolox/utils/boxes.py",4471,"65d8341d67ec35d65f4a73ba8480ce536a8bb38f4b4088f99db82bc812ec9bcb"),
]

PKG_ROOT=WORK_ROOT/"yolox_upstream"
PKG=PKG_ROOT/"yolox_ref"
PKG.mkdir(parents=True,exist_ok=True)
(PKG/"__init__.py").write_text("",encoding="utf-8")

def fetch_bytes(path):
    request=urllib.request.Request(RAW_BASE+path,headers={"User-Agent":"DIMER-workshop/1.0"})
    with urllib.request.urlopen(request,timeout=60) as response:
        return response.read()

for target,(source,size,digest) in UPSTREAM_FILES.items():
    payload=fetch_bytes(source)
    if len(payload)!=size or hashlib.sha256(payload).hexdigest()!=digest:
        raise ValueError(f"YOLOX upstream integrity mismatch: {source}")
    text=payload.decode("utf-8")
    if target=="yolo_head.py":
        text=text.replace("from loguru import logger\n","import warnings\n")
        text=text.replace("from yolox.utils import bboxes_iou, meshgrid","from .ops import bboxes_iou, meshgrid")
        text=text.replace("logger.error(","warnings.warn(")
    (PKG/target).write_text(text,encoding="utf-8")

ops_parts=[]
for source,size,digest in OPS_FILES:
    payload=fetch_bytes(source)
    if len(payload)!=size or hashlib.sha256(payload).hexdigest()!=digest:
        raise ValueError(f"YOLOX upstream integrity mismatch: {source}")
    lines=payload.decode("utf-8").splitlines()
    while lines and lines[0].startswith("#!"):
        lines=lines[1:]
    ops_parts.append("\n".join(lines))
(PKG/"ops.py").write_text(
    '"""Minimal YOLOX utility subset pinned to upstream."""\n\n'+"\n\n".join(ops_parts),
    encoding="utf-8",
)

if str(PKG_ROOT.resolve()) not in sys.path:
    sys.path.insert(0,str(PKG_ROOT.resolve()))

from yolox_ref.coco_classes import COCO_CLASSES
from yolox_ref.yolo_pafpn import YOLOPAFPN
from yolox_ref.yolo_head import YOLOXHead
from yolox_ref.yolox import YOLOX
from yolox_ref.ops import postprocess

print({
    "upstream_revision":YOLOX_CODE_REVISION,
    "modules":sorted(p.name for p in PKG.glob("*.py")),
    "release_gate":"carry verified source inline before promotion",
})

# 9. Common detector wrappers

Both wrappers expose one normalized interface:

- custom three-class re-head;
- low-threshold evaluation predictions;
- bounded model-native adaptation;
- export/reload.

The evaluator above remains independent of both wrappers.

In [ ]:
# @title RT-DETR reference wrapper
import math
import copy
from transformers import AutoImageProcessor, RTDetrForObjectDetection

class RTDETRWorkshop:
    def __init__(self, model, processor, class_names, device, adapted=False):
        self.model=model
        self.processor=processor
        self.class_names=tuple(class_names)
        self.device=device
        self.adapted=adapted

    @classmethod
    def from_pretrained(cls, class_names=CLASS_NAMES, seed=0, device=DEVICE):
        torch.manual_seed(seed)
        processor=AutoImageProcessor.from_pretrained(
            str(RTDETR_ROOT),local_files_only=True,trust_remote_code=False
        )
        model=RTDetrForObjectDetection.from_pretrained(
            str(RTDETR_ROOT),
            local_files_only=True,
            num_labels=len(class_names),
            ignore_mismatched_sizes=True,
            trust_remote_code=False,
            use_pretrained_backbone=False,
        )
        prior=-math.log((1-0.01)/0.01)
        torch.nn.init.constant_(model.model.enc_score_head.bias,prior)
        for embed in model.model.decoder.class_embed:
            torch.nn.init.constant_(embed.bias,prior)
        model=model.to(device).eval()
        return cls(model,processor,class_names,device)

    def detect(self,image,threshold=EVAL_SCORE_THRESHOLD):
        rgb=image.convert("RGB")
        inputs=self.processor(images=rgb,return_tensors="pt").to(self.device)
        was_training=self.model.training
        self.model.eval()
        with torch.inference_mode():
            outputs=self.model(**inputs)
        if was_training:self.model.train()
        result=self.processor.post_process_object_detection(
            outputs,threshold=threshold,target_sizes=[(rgb.height,rgb.width)]
        )[0]
        dets=[]
        for box,label,score in zip(result["boxes"],result["labels"],result["scores"]):
            idx=int(label)
            if idx<len(self.class_names):
                dets.append({
                    "box":[float(v) for v in box.tolist()],
                    "label":self.class_names[idx],
                    "score":float(score),
                })
        return sorted(dets,key=lambda d:-d["score"])

    def predict_many(self,records,threshold=EVAL_SCORE_THRESHOLD):
        return [self.detect(r["image"],threshold)[:MAX_EVAL_DETECTIONS] for r in records]

    def finetune(self,records,epochs=3,batch_size=4,learning_rate=1e-4,seed=0):
        torch.manual_seed(seed)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
        rng=np.random.default_rng(seed)

        for p in self.model.model.backbone.parameters():
            p.requires_grad=False
        trainable=[p for p in self.model.parameters() if p.requires_grad]
        optimizer=torch.optim.AdamW(trainable,lr=learning_rate,weight_decay=1e-4)
        class_to_id={name:i for i,name in enumerate(self.class_names)}
        history=[]

        for epoch in range(epochs):
            self.model.train()
            order=rng.permutation(len(records))
            batch_losses=[]
            for start in range(0,len(records),batch_size):
                batch=[records[i] for i in order[start:start+batch_size]]
                images=[r["image"].convert("RGB") for r in batch]
                labels=[]
                for r in batch:
                    w,h=r["image"].size
                    boxes=[]
                    for x0,y0,x1,y1 in r["boxes"]:
                        boxes.append([
                            (x0+x1)/2/w,(y0+y1)/2/h,
                            (x1-x0)/w,(y1-y0)/h,
                        ])
                    labels.append({
                        "class_labels":torch.tensor(
                            [class_to_id[x] for x in r["labels"]],
                            dtype=torch.long,device=self.device
                        ),
                        "boxes":torch.tensor(boxes,dtype=torch.float32,device=self.device),
                    })
                inputs=self.processor(images=images,return_tensors="pt").to(self.device)
                optimizer.zero_grad(set_to_none=True)
                output=self.model(pixel_values=inputs["pixel_values"],labels=labels)
                loss=output.loss
                loss.backward()
                optimizer.step()
                batch_losses.append(float(loss.detach().cpu()))
            row={"epoch":epoch+1,"loss":float(np.mean(batch_losses))}
            history.append(row);print("RT-DETR",row)
        self.model.eval();self.adapted=True
        return {
            "epochs":epochs,"batch_size":batch_size,"learning_rate":learning_rate,
            "optimizer":"AdamW(weight_decay=1e-4)",
            "loss":"RT-DETR native Varifocal + L1 + GIoU",
            "freeze_backbone":True,
            "trainable_parameters":sum(p.numel() for p in trainable),
            "total_parameters":sum(p.numel() for p in self.model.parameters()),
            "history":history,
        }

    def save_artifact(self,path):
        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
        payload={
            "format":"rtdetr-adapter-v1",
            "model_id":RTDETR_SPEC["model_id"],
            "revision":RTDETR_SPEC["revision"],
            "class_names":list(self.class_names),
            "adapted":self.adapted,
            "state_dict":{k:v.detach().cpu() for k,v in self.model.state_dict().items()},
        }
        torch.save(payload,path)
        return {"path":str(path),"bytes":path.stat().st_size,"sha256":sha256_file(path)}

    @classmethod
    def load_artifact(cls,path,device=DEVICE):
        payload=torch.load(path,map_location="cpu",weights_only=True)
        if payload["format"]!="rtdetr-adapter-v1":
            raise ValueError("foreign RT-DETR artifact")
        if payload["revision"]!=RTDETR_SPEC["revision"]:
            raise ValueError("RT-DETR artifact base revision mismatch")
        pipe=cls.from_pretrained(tuple(payload["class_names"]),seed=0,device=device)
        pipe.model.load_state_dict(payload["state_dict"],strict=True)
        pipe.model.eval();pipe.adapted=True
        return pipe

In [ ]:
# @title Generic YOLOX-S / YOLOX-X reference wrapper
INPUT_SIZE=(640,640)
PAD_VALUE=114
YOLOX_EVAL_NMS=0.65
MAX_LABELS_PER_IMAGE=6

def stage_yolox_checkpoint(key):
    spec=YOLOX_VARIANTS[key]
    target=WORK_ROOT/"yolox_weights"/spec["checkpoint"]
    if not (target.is_file() and target.stat().st_size==spec["bytes"] and sha256_file(target)==spec["sha256"]):
        request=urllib.request.Request(spec["url"],headers={"User-Agent":"DIMER-workshop/1.0"})
        with urllib.request.urlopen(request,timeout=120) as response:
            payload=response.read()
        if len(payload)!=spec["bytes"] or hashlib.sha256(payload).hexdigest()!=spec["sha256"]:
            raise ValueError(f"{key}: checkpoint integrity mismatch")
        target.write_bytes(payload)
    return target

def yolox_preprocess(image):
    image=image.convert("RGB")
    width,height=image.size
    ratio=min(INPUT_SIZE[0]/height,INPUT_SIZE[1]/width)
    new_w,new_h=int(width*ratio),int(height*ratio)
    canvas=np.full((INPUT_SIZE[0],INPUT_SIZE[1],3),PAD_VALUE,dtype=np.uint8)
    resized=image if (new_w,new_h)==(width,height) else image.resize((new_w,new_h),Image.BILINEAR)
    canvas[:new_h,:new_w]=np.asarray(resized,dtype=np.uint8)
    chw=canvas.transpose(2,0,1)[::-1] # RGB -> BGR
    return np.ascontiguousarray(chw,dtype=np.float32),ratio

def build_yolox(depth,width,num_classes):
    backbone=YOLOPAFPN(depth=depth,width=width,in_channels=[256,512,1024],act="silu")
    head=YOLOXHead(num_classes=num_classes,width=width,in_channels=[256,512,1024],act="silu")
    head.initialize_biases(1e-2)
    return YOLOX(backbone,head)

class YOLOXWorkshop:
    def __init__(self,key,model,class_names,device,adapted=False):
        self.key=key;self.model=model;self.class_names=tuple(class_names)
        self.device=device;self.adapted=adapted

    @classmethod
    def from_pretrained(cls,key,class_names=CLASS_NAMES,seed=0,device=DEVICE):
        spec=YOLOX_VARIANTS[key]
        path=stage_yolox_checkpoint(key)
        torch.manual_seed(seed)
        model=build_yolox(spec["depth"],spec["width"],len(class_names))
        ckpt=torch.load(path,map_location="cpu",weights_only=True)
        base=ckpt["model"]
        if tuple(class_names)==tuple(COCO_CLASSES):
            model.load_state_dict(base,strict=True)
        else:
            current=model.state_dict()
            compatible={
                k:v for k,v in base.items()
                if k in current and tuple(v.shape)==tuple(current[k].shape)
            }
            result=model.load_state_dict(compatible,strict=False)
            bad=[
                k for k in result.missing_keys
                if not k.startswith("head.cls_preds.")
            ]
            if bad or result.unexpected_keys:
                raise RuntimeError({
                    "unexpected_missing":bad,
                    "unexpected_keys":result.unexpected_keys,
                })
        model=model.to(device).eval()
        return cls(key,model,class_names,device)

    def detect(self,image,threshold=EVAL_SCORE_THRESHOLD,nms_threshold=YOLOX_EVAL_NMS):
        chw,ratio=yolox_preprocess(image)
        tensor=torch.from_numpy(chw).unsqueeze(0).to(self.device)
        was_training=self.model.training
        self.model.eval()
        with torch.no_grad():
            raw=self.model(tensor)
        kept=postprocess(
            raw,len(self.class_names),
            conf_thre=threshold,nms_thre=nms_threshold,class_agnostic=False
        )[0]
        if was_training:self.model.train()
        if kept is None:return []
        dets=[]
        for x0,y0,x1,y1,obj,cls_conf,cls_id in kept.tolist():
            idx=int(cls_id)
            dets.append({
                "box":[x0/ratio,y0/ratio,x1/ratio,y1/ratio],
                "label":self.class_names[idx],
                "score":float(obj*cls_conf),
            })
        return sorted(dets,key=lambda d:-d["score"])

    def predict_many(self,records,threshold=EVAL_SCORE_THRESHOLD):
        return [
            self.detect(r["image"],threshold,YOLOX_EVAL_NMS)[:MAX_EVAL_DETECTIONS]
            for r in records
        ]

    def _as_batch(self,records):
        batch=torch.zeros(len(records),3,640,640)
        labels=torch.zeros(len(records),MAX_LABELS_PER_IMAGE,5)
        for i,r in enumerate(records):
            chw,ratio=yolox_preprocess(r["image"])
            batch[i]=torch.from_numpy(chw)
            for j,(box,label) in enumerate(zip(r["boxes"],r["labels"])):
                x0,y0,x1,y1=(float(v)*ratio for v in box)
                labels[i,j]=torch.tensor([
                    float(self.class_names.index(label)),
                    (x0+x1)/2,(y0+y1)/2,x1-x0,y1-y0,
                ])
        return batch,labels

    def finetune(self,records,epochs=6,batch_size=2,learning_rate=1e-3,seed=0):
        torch.manual_seed(seed)
        rng=np.random.default_rng(seed)
        images,targets=self._as_batch(records)
        images,targets=images.to(self.device),targets.to(self.device)

        for p in self.model.backbone.parameters():
            p.requires_grad=False
        self.model.train()
        self.model.backbone.eval()
        self.model.head.use_l1=False
        trainable=[p for p in self.model.parameters() if p.requires_grad]
        optimizer=torch.optim.SGD(
            trainable,lr=learning_rate,momentum=0.9,weight_decay=5e-4
        )
        history=[]
        for epoch in range(epochs):
            order=rng.permutation(len(records))
            rows=[]
            for start in range(0,len(records),batch_size):
                idx=torch.as_tensor(order[start:start+batch_size].copy(),dtype=torch.long,device=self.device)
                optimizer.zero_grad(set_to_none=True)
                losses=self.model(images[idx],targets[idx])
                losses["total_loss"].backward()
                optimizer.step()
                rows.append({
                    k:(float(v.detach()) if torch.is_tensor(v) else float(v))
                    for k,v in losses.items()
                })
            row={"epoch":epoch+1,**{
                k:float(np.mean([r[k] for r in rows])) for k in rows[0]
            }}
            history.append(row);print(self.key,row)
        self.model.eval();self.adapted=True
        return {
            "epochs":epochs,"batch_size":batch_size,"learning_rate":learning_rate,
            "optimizer":"SGD(momentum=0.9, weight_decay=5e-4)",
            "loss":"YOLOX SimOTA (IoU + objectness + classification)",
            "freeze_backbone":True,
            "trainable_parameters":sum(p.numel() for p in trainable),
            "total_parameters":sum(p.numel() for p in self.model.parameters()),
            "history":history,
        }

    def save_artifact(self,path):
        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
        spec=YOLOX_VARIANTS[self.key]
        payload={
            "format":"dimer-yolox-detection-adapter/1",
            "variant":self.key,
            "release":"0.1.1rc0",
            "upstream_code_revision":YOLOX_CODE_REVISION,
            "base_sha256":spec["sha256"],
            "class_names":list(self.class_names),
            "adapted":self.adapted,
            "state_dict":{k:v.detach().cpu() for k,v in self.model.state_dict().items()},
        }
        torch.save(payload,path)
        return {"path":str(path),"bytes":path.stat().st_size,"sha256":sha256_file(path)}

    @classmethod
    def load_artifact(cls,path,device=DEVICE):
        payload=torch.load(path,map_location="cpu",weights_only=True)
        if payload["format"]!="dimer-yolox-detection-adapter/1":
            raise ValueError("foreign YOLOX artifact")
        key=payload["variant"]
        if payload["base_sha256"]!=YOLOX_VARIANTS[key]["sha256"]:
            raise ValueError("YOLOX artifact base digest mismatch")
        pipe=cls.from_pretrained(key,tuple(payload["class_names"]),seed=0,device=device)
        pipe.model.load_state_dict(payload["state_dict"],strict=True)
        pipe.model.eval();pipe.adapted=True
        return pipe

# 10. Pre-adaptation transfer baseline on validation

Each detector is re-headed onto the new three-class vocabulary **before training**.

This asks:

> How much useful localization structure transfers from COCO before the new classification head has learned the target labels?

The empty detector remains the task floor.

In [ ]:
# @title Re-head core detectors and score validation before training
import time
import gc

pre_validation_predictions={}
runtime_records={}
pipes={}

for key in MODEL_KEYS:
    t0=time.perf_counter()
    if key=="rtdetr":
        pipe=RTDETRWorkshop.from_pretrained(CLASS_NAMES,seed=0,device=DEVICE)
    else:
        pipe=YOLOXWorkshop.from_pretrained(key,CLASS_NAMES,seed=0,device=DEVICE)
    load_seconds=time.perf_counter()-t0
    t0=time.perf_counter()
    preds=pipe.predict_many(validation_records,EVAL_SCORE_THRESHOLD)
    infer_seconds=time.perf_counter()-t0

    pre_validation_predictions[key]=preds
    pipes[key]=pipe
    runtime_records[key]={
        "load_seconds":load_seconds,
        "pre_validation_inference_seconds":infer_seconds,
    }

pre_val_rows=[]
for key,preds in pre_validation_predictions.items():
    m=average_precision(preds,validation_records,CLASS_NAMES)
    op=operating_metrics(preds,validation_records,DISPLAY_THRESHOLD)
    pre_val_rows.append({"model":key,**m,**op})
pre_val_table=pd.DataFrame(pre_val_rows)
display(pre_val_table)
pre_val_table.to_csv(OUTPUT_ROOT/"pre_adaptation"/"validation_metrics.csv",index=False)

# 11. Model-native bounded adaptation

Fairness does **not** mean forcing unrelated detectors to use one loss or optimizer.

### RT-DETR

- 3 epochs
- batch 4
- AdamW, `1e-4`
- ResNet backbone frozen
- native Varifocal + L1 + GIoU objective

### YOLOX

- 6 epochs
- batch 2
- SGD + momentum 0.9, `1e-3`
- backbone + PAFPN frozen
- native SimOTA / IoU / objectness / class objective

Every model trains on the exact same 36 images and boxes.

In [ ]:
# @title Fine-tune selected detectors
training_reports={}
artifact_descriptors={}

for key in MODEL_KEYS:
    pipe=pipes[key]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()

    if key=="rtdetr":
        report=pipe.finetune(
            train_records,
            epochs=RTDETR_EPOCHS,
            batch_size=RTDETR_BATCH_SIZE,
            learning_rate=RTDETR_LEARNING_RATE,
            seed=0,
        )
    else:
        report=pipe.finetune(
            train_records,
            epochs=YOLOX_EPOCHS,
            batch_size=YOLOX_BATCH_SIZE,
            learning_rate=YOLOX_LEARNING_RATE,
            seed=0,
        )

    train_seconds=time.perf_counter()-t0
    report["training_seconds"]=train_seconds
    report["peak_cuda_bytes"]=(
        int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None
    )
    training_reports[key]=report

    artifact_dir=OUTPUT_ROOT/"artifacts"/key
    artifact_dir.mkdir(parents=True,exist_ok=True)
    artifact_path=artifact_dir/"adapter.pt"
    artifact_descriptors[key]=pipe.save_artifact(artifact_path)

    (OUTPUT_ROOT/"adaptation"/f"{key}_training.json").write_text(
        json.dumps(report,indent=2),encoding="utf-8"
    )

display(pd.DataFrame([
    {
        "model":key,
        "trainable_parameters":r["trainable_parameters"],
        "total_parameters":r["total_parameters"],
        "training_seconds":r["training_seconds"],
        "peak_vram_MiB":(
            r["peak_cuda_bytes"]/2**20 if r["peak_cuda_bytes"] is not None else None
        ),
        "artifact_MiB":artifact_descriptors[key]["bytes"]/2**20,
    }
    for key,r in training_reports.items()
]))

# 12. Post-adaptation validation and threshold sensitivity

In [ ]:
# @title Validation after adaptation
post_validation_predictions={}
post_val_rows=[]

for key,pipe in pipes.items():
    preds=pipe.predict_many(validation_records,EVAL_SCORE_THRESHOLD)
    post_validation_predictions[key]=preds
    m=average_precision(preds,validation_records,CLASS_NAMES)
    op=operating_metrics(preds,validation_records,DISPLAY_THRESHOLD)
    post_val_rows.append({"model":key,**m,**op})

post_val_table=pd.DataFrame(post_val_rows)
display(post_val_table)
post_val_table.to_csv(OUTPUT_ROOT/"validation"/"metrics.csv",index=False)

validation_compare=pre_val_table[["model","ap","ap50","ap75"]].merge(
    post_val_table[["model","ap","ap50","ap75"]],
    on="model",suffixes=("_pre","_post")
)
display(validation_compare)

In [ ]:
# @title Validation-only display-threshold sweep
threshold_rows=[]
for key,pipe in pipes.items():
    for threshold in (0.05,0.10,0.20,0.30,0.50,0.70):
        if key=="rtdetr":
            predictions=[
                pipe.detect(r["image"],threshold)[:MAX_EVAL_DETECTIONS]
                for r in validation_records
            ]
        else:
            predictions=[
                pipe.detect(r["image"],threshold,YOLOX_EVAL_NMS)[:MAX_EVAL_DETECTIONS]
                for r in validation_records
            ]
        values=operating_metrics(predictions,validation_records,threshold,iou=0.50)
        threshold_rows.append({"model":key,"threshold":threshold,**values})

threshold_table=pd.DataFrame(threshold_rows)
threshold_table.to_csv(OUTPUT_ROOT/"validation"/"threshold_sweep.csv",index=False)
display(threshold_table)

# 13. Freeze before opening the test split

At this point:

- dataset and splits are fixed;
- model identities are fixed;
- model-native training recipes are fixed;
- evaluation cutoff and IoU thresholds are fixed;
- artifacts have been exported;
- validation has been inspected.

Nothing may now be changed in response to the test results.

In [ ]:
# @title Freeze experiment
frozen_experiment={
    "notebook_spec":"2.1",
    "profile":"E2E",
    "mode":"WORKSHOP",
    "execution_tier":WORKSHOP_TIER,
    "dataset":dataset_manifest,
    "models":{
        "rtdetr":{
            "id":RTDETR_SPEC["model_id"],
            "revision":RTDETR_SPEC["revision"],
            "weight_sha256":RTDETR_SPEC["files"]["model.safetensors"][1],
            "adaptation":{
                "epochs":RTDETR_EPOCHS,"batch_size":RTDETR_BATCH_SIZE,
                "learning_rate":RTDETR_LEARNING_RATE,
                "optimizer":"AdamW(weight_decay=1e-4)",
                "freeze_backbone":True,
            },
        },
        **{
            key:{
                "id":"Megvii-BaseDetection/YOLOX",
                "release":"0.1.1rc0",
                "upstream_code_revision":YOLOX_CODE_REVISION,
                "weight_sha256":YOLOX_VARIANTS[key]["sha256"],
                "adaptation":{
                    "epochs":YOLOX_EPOCHS,"batch_size":YOLOX_BATCH_SIZE,
                    "learning_rate":YOLOX_LEARNING_RATE,
                    "optimizer":"SGD(momentum=0.9, weight_decay=5e-4)",
                    "freeze_backbone":True,
                },
            }
            for key in MODEL_KEYS if key.startswith("yolox_")
        },
    },
    "evaluation":{
        "score_cutoff":EVAL_SCORE_THRESHOLD,
        "display_threshold":DISPLAY_THRESHOLD,
        "yolox_nms":YOLOX_EVAL_NMS,
        "max_detections":MAX_EVAL_DETECTIONS,
        "iou_thresholds":list(IOU_THRESHOLDS),
        "metric":"notebook-owned class-aware 101-point interpolated AP",
    },
    "artifacts":artifact_descriptors,
    "validation":{
        "pre":pre_val_table.to_dict("records"),
        "post":post_val_table.to_dict("records"),
    },
    "candidate_release_gate":(
        "carry the pinned upstream YOLOX source modules inline instead of fetching them at runtime"
    ),
}

freeze_path=OUTPUT_ROOT/"frozen"/"frozen_experiment.json"
freeze_path.write_text(json.dumps(frozen_experiment,indent=2),encoding="utf-8")
print("Frozen:",freeze_path)

# 14. Independent test: pre-adaptation vs adapted

The test is opened only now.

For the pre-adaptation rows, fresh re-headed detectors are reconstructed with the same deterministic seed.

For adapted rows, the exported artifacts are reloaded into fresh objects before scoring.

This also turns the test into an artifact-boundary check.

In [ ]:
# @title Fresh reconstruction and test evaluation
del pipes
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

test_predictions={}
test_rows=[]
fresh_adapted={}

for key in MODEL_KEYS:
    # Deterministic pre-adaptation detector.
    if key=="rtdetr":
        pre=RTDETRWorkshop.from_pretrained(CLASS_NAMES,seed=0,device=DEVICE)
    else:
        pre=YOLOXWorkshop.from_pretrained(key,CLASS_NAMES,seed=0,device=DEVICE)

    t0=time.perf_counter()
    pre_preds=pre.predict_many(test_records,EVAL_SCORE_THRESHOLD)
    pre_seconds=time.perf_counter()-t0
    pre_m=average_precision(pre_preds,test_records,CLASS_NAMES)
    pre_op=operating_metrics(pre_preds,test_records,DISPLAY_THRESHOLD)
    test_rows.append({
        "model":key,"state":"pre","inference_seconds":pre_seconds,
        "ms_per_image":1000*pre_seconds/len(test_records),**pre_m,**pre_op
    })
    test_predictions[(key,"pre")]=pre_preds

    del pre
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

    artifact_path=OUTPUT_ROOT/"artifacts"/key/"adapter.pt"
    if key=="rtdetr":
        adapted=RTDETRWorkshop.load_artifact(artifact_path,DEVICE)
    else:
        adapted=YOLOXWorkshop.load_artifact(artifact_path,DEVICE)

    t0=time.perf_counter()
    adapted_preds=adapted.predict_many(test_records,EVAL_SCORE_THRESHOLD)
    adapted_seconds=time.perf_counter()-t0
    adapted_m=average_precision(adapted_preds,test_records,CLASS_NAMES)
    adapted_op=operating_metrics(adapted_preds,test_records,DISPLAY_THRESHOLD)
    test_rows.append({
        "model":key,"state":"adapted","inference_seconds":adapted_seconds,
        "ms_per_image":1000*adapted_seconds/len(test_records),**adapted_m,**adapted_op
    })
    test_predictions[(key,"adapted")]=adapted_preds
    fresh_adapted[key]=adapted

test_table=pd.DataFrame(test_rows)
display(test_table)
test_table.to_csv(OUTPUT_ROOT/"test"/"aggregate_metrics.csv",index=False)

## Read AP50 and AP75 together

A synthetic task can make AP50 saturate quickly.

AP75 and the mean AP over 0.50–0.95 reveal whether localization remains tight when the overlap requirement becomes stricter.

In [ ]:
# @title Per-class AP50 table
per_class_rows=[]
for (key,state),preds in test_predictions.items():
    m=average_precision(preds,test_records,CLASS_NAMES)
    for cls,value in m["per_class_ap50"].items():
        per_class_rows.append({
            "model":key,"state":state,"class":cls,"ap50":value
        })
per_class_table=pd.DataFrame(per_class_rows)
display(per_class_table)
per_class_table.to_csv(OUTPUT_ROOT/"test"/"per_class_metrics.csv",index=False)

# 15. Object-size and scene-density diagnostics

These are tutorial diagnostics, **not COCO's official small/medium/large area buckets**.

Size groups are defined from bounding-box area relative to the 640×640 image.

Density groups use one, two, or three objects per scene.

In [ ]:
# @title Size and density recall diagnostics
def recall_for_subset(predictions,records,selector,iou=0.5,threshold=DISPLAY_THRESHOLD):
    hits=total=0
    for dets,ref in zip(predictions,records):
        selected=[
            (box,label) for box,label in zip(ref["boxes"],ref["labels"])
            if selector(box,ref)
        ]
        total+=len(selected)
        claimed=[False]*len(selected)
        for det in sorted((d for d in dets if d["score"]>=threshold),key=lambda x:-x["score"]):
            best,best_iou=-1,0
            for j,(box,label) in enumerate(selected):
                if claimed[j] or label!=det["label"]:continue
                value=box_iou(det["box"],box)
                if value>best_iou:best,best_iou=j,value
            if best>=0 and best_iou>=iou:
                claimed[best]=True
        hits+=sum(claimed)
    return hits/total if total else np.nan,total

def area_fraction(box):
    x0,y0,x1,y1=box
    return ((x1-x0)*(y1-y0))/(640*640)

size_selectors={
    "small":lambda box,ref:area_fraction(box)<0.020,
    "medium":lambda box,ref:0.020<=area_fraction(box)<0.040,
    "large":lambda box,ref:area_fraction(box)>=0.040,
}

diag_rows=[]
for key in MODEL_KEYS:
    preds=test_predictions[(key,"adapted")]
    for group,selector in size_selectors.items():
        recall,n=recall_for_subset(preds,test_records,selector)
        diag_rows.append({"model":key,"dimension":"size","group":group,"recall50":recall,"objects":n})
    for count in (1,2,3):
        mask=[len(r["boxes"])==count for r in test_records]
        subset_records=[r for r,m in zip(test_records,mask) if m]
        subset_preds=[p for p,m in zip(preds,mask) if m]
        if subset_records:
            op=operating_metrics(subset_preds,subset_records,DISPLAY_THRESHOLD)
            diag_rows.append({
                "model":key,"dimension":"scene_density","group":str(count),
                "recall50":op["recall50"],"objects":sum(len(r["boxes"]) for r in subset_records)
            })

diagnostics=pd.DataFrame(diag_rows)
display(diagnostics)
diagnostics.to_csv(OUTPUT_ROOT/"test"/"size_density_metrics.csv",index=False)

# 16. Deterministic disagreement and error gallery

In [ ]:
# @title Per-image adapted metrics for disagreement analysis
def image_ap50(dets,record):
    return average_precision([dets],[record],CLASS_NAMES,iou_thresholds=(0.5,))["ap50"]

pair_rows=[]
for idx,record in enumerate(test_records):
    row={"image_id":record["id"],"n_objects":len(record["boxes"])}
    for key in ("rtdetr","yolox_s"):
        if key in MODEL_KEYS:
            dets=test_predictions[(key,"adapted")][idx]
            row[f"{key}_ap50"]=image_ap50(dets,record)
            row[f"{key}_detections"]=sum(d["score"]>=DISPLAY_THRESHOLD for d in dets)
    if "rtdetr" in MODEL_KEYS and "yolox_s" in MODEL_KEYS:
        row["rtdetr_advantage"]=row["rtdetr_ap50"]-row["yolox_s_ap50"]
        row["detection_count_difference"]=row["rtdetr_detections"]-row["yolox_s_detections"]
    pair_rows.append(row)

disagreements=pd.DataFrame(pair_rows)
disagreements.to_csv(OUTPUT_ROOT/"test"/"disagreements.csv",index=False)
display(disagreements.sort_values("rtdetr_advantage",ascending=False).head())
display(disagreements.sort_values("rtdetr_advantage").head())

In [ ]:
# @title Plot deterministic comparison examples
if "rtdetr" in MODEL_KEYS and "yolox_s" in MODEL_KEYS:
    candidates=[
        ("RT-DETR advantage",int(disagreements["rtdetr_advantage"].idxmax())),
        ("YOLOX-S advantage",int(disagreements["rtdetr_advantage"].idxmin())),
        ("Three-object scene",next((i for i,r in enumerate(test_records) if len(r["boxes"])==3),0)),
    ]

    for title,idx in candidates:
        record=test_records[idx]
        fig,axes=plt.subplots(1,3,figsize=(15,5))

        for ax in axes:
            ax.imshow(record["image"]);ax.axis("off")

        axes[0].set_title("Ground truth")
        for box,label in zip(record["boxes"],record["labels"]):
            x0,y0,x1,y1=box
            axes[0].add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=2))
            axes[0].text(x0,y0,label,fontsize=7)

        for ax,key in zip(axes[1:],("rtdetr","yolox_s")):
            ax.set_title(key)
            for det in test_predictions[(key,"adapted")][idx]:
                if det["score"]<DISPLAY_THRESHOLD:continue
                x0,y0,x1,y1=det["box"]
                ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=2))
                ax.text(x0,y0,f"{det['label']} {det['score']:.2f}",fontsize=7)

        fig.suptitle(f"{title}: {record['id']}")
        plt.tight_layout()
        safe=title.lower().replace(" ","_")
        plt.savefig(OUTPUT_ROOT/"figures"/f"gallery_{safe}.png",dpi=150,bbox_inches="tight")
        plt.show()

# 17. Computational tradeoffs

No composite winner is calculated.

The practical comparison includes:

- base checkpoint size;
- total parameters;
- trainable parameters;
- training wall time;
- test inference latency;
- artifact size;
- AP/AP75.

YOLOX-X in `FULL` is specifically a **scale experiment**: larger is not assumed to be better on this small transfer task.

In [ ]:
# @title Tradeoff table
tradeoff_rows=[]
for key in MODEL_KEYS:
    adapted=test_table[(test_table["model"]==key)&(test_table["state"]=="adapted")].iloc[0]
    report=training_reports[key]
    if key=="rtdetr":
        weight_bytes=RTDETR_SPEC["files"]["model.safetensors"][0]
    else:
        weight_bytes=YOLOX_VARIANTS[key]["bytes"]
    tradeoff_rows.append({
        "model":key,
        "weight_MiB":weight_bytes/2**20,
        "parameters_M":report["total_parameters"]/1e6,
        "trainable_M":report["trainable_parameters"]/1e6,
        "training_seconds":report["training_seconds"],
        "ms_per_test_image":adapted["ms_per_image"],
        "artifact_MiB":artifact_descriptors[key]["bytes"]/2**20,
        "test_ap":adapted["ap"],
        "test_ap75":adapted["ap75"],
    })

tradeoffs=pd.DataFrame(tradeoff_rows)
display(tradeoffs)
tradeoffs.to_csv(OUTPUT_ROOT/"test"/"computational_tradeoffs.csv",index=False)

# 18. Fresh reload parity

Test evaluation above already used freshly reconstructed artifact-backed models.

The next cell additionally verifies one fixed validation image at the raw-output boundary:

- same detection count;
- same class labels;
- box coordinates within tolerance;
- scores within tolerance.

In [ ]:
# @title Artifact reload parity on a fixed validation image
parity={}
verify_record=validation_records[0]

for key in MODEL_KEYS:
    # `fresh_adapted` was reconstructed from the artifact before test.
    reloaded=fresh_adapted[key]

    # Load again to create a distinct fresh object.
    artifact_path=OUTPUT_ROOT/"artifacts"/key/"adapter.pt"
    if key=="rtdetr":
        again=RTDETRWorkshop.load_artifact(artifact_path,DEVICE)
    else:
        again=YOLOXWorkshop.load_artifact(artifact_path,DEVICE)

    a=reloaded.detect(verify_record["image"],DISPLAY_THRESHOLD)
    b=again.detect(verify_record["image"],DISPLAY_THRESHOLD)

    if len(a)!=len(b):
        raise RuntimeError(f"{key}: reload detection count mismatch")
    max_box=max_score=0.0
    for da,db in zip(a,b):
        if da["label"]!=db["label"]:
            raise RuntimeError(f"{key}: reload label mismatch")
        max_box=max(max_box,max(abs(x-y) for x,y in zip(da["box"],db["box"])))
        max_score=max(max_score,abs(da["score"]-db["score"]))

    parity[key]={
        "n_detections":len(a),
        "max_abs_box_difference":max_box,
        "max_abs_score_difference":max_score,
    }
    if max_box>1e-4 or max_score>1e-6:
        raise RuntimeError(f"{key}: reload parity exceeded tolerance")

print(parity)

# 19. New-seed inference sanity check

Three synthetic scenes are generated using seed `2026`, completely outside the train/validation/test fixture.

Their exact boxes are known, so same-label IoU can be shown as an additional sanity check.

These examples do **not** enter the main test metrics.

In [ ]:
# @title New-seed examples
new_records=generate_sign_dataset(3,NEW_DATA_SEED,MAX_OBJECTS)

new_rows=[]
for key in MODEL_KEYS:
    model=fresh_adapted[key]
    preds=model.predict_many(new_records,EVAL_SCORE_THRESHOLD)
    op=operating_metrics(preds,new_records,DISPLAY_THRESHOLD)
    new_rows.append({"model":key,**op})
    for i,(record,dets) in enumerate(zip(new_records,preds)):
        payload={
            "image_id":record["id"],
            "detections":[d for d in dets if d["score"]>=DISPLAY_THRESHOLD],
        }
        (OUTPUT_ROOT/"new_data"/f"{key}_{i}.json").write_text(
            json.dumps(payload,indent=2),encoding="utf-8"
        )

display(pd.DataFrame(new_rows))

# 20. BYOD

Optional BYOD expects a ZIP with:

```text
dataset.zip
├── annotations.csv
└── images/
    ├── image001.jpg
    └── ...
```

`annotations.csv`:

```text
image_id,file,label,x0,y0,x1,y1,split
img001,images/image001.jpg,class_a,10,20,100,150,train
...
```

Preferred explicit split values:

- `train`
- `validation`
- `test`

If omitted, a deterministic 60/20/20 **image-level** split should be created and then class coverage checked.

The notebook MUST refuse a split if a declared class is absent from training, validation, or test.

### Geometry caveat

The canonical 640×640 fixture removes preprocessing geometry differences.

Arbitrary BYOD does not:

- RT-DETR squashes to 640×640;
- YOLOX preserves aspect ratio and letterboxes.

BYOD therefore compares the **complete deployed detector systems**, not architecture alone.

In [ ]:
# @title BYOD loader (optional branch)
import csv
import zipfile
import shutil

def safe_extract_zip(zip_path,destination):
    destination=Path(destination).resolve()
    if destination.exists():shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as archive:
        infos=archive.infolist()
        if sum(i.file_size for i in infos)>2*1024**3:
            raise ValueError("expanded BYOD ZIP exceeds 2 GiB")
        for info in infos:
            rel=Path(info.filename)
            if rel.is_absolute() or ".." in rel.parts:
                raise ValueError(f"unsafe ZIP path {info.filename}")
            mode=info.external_attr>>16
            if mode and (mode&0o170000)==0o120000:
                raise ValueError(f"symlink member refused: {info.filename}")
        archive.extractall(destination)
    return destination

def load_byod(zip_path):
    root=safe_extract_zip(zip_path,WORK_ROOT/"byod")
    candidates=list(root.rglob("annotations.csv"))
    if len(candidates)!=1:
        raise ValueError("BYOD ZIP must contain exactly one annotations.csv")
    table=pd.read_csv(candidates[0])
    required={"image_id","file","label","x0","y0","x1","y1"}
    if not required<=set(table.columns):
        raise ValueError(f"annotations.csv missing {sorted(required-set(table.columns))}")

    records={}
    base=candidates[0].parent
    for row in table.to_dict("records"):
        rid=str(row["image_id"])
        path=(base/str(row["file"])).resolve()
        if base.resolve() not in path.parents:
            raise ValueError("annotation file path escapes dataset root")
        if not path.is_file():
            raise ValueError(f"missing image {path}")
        if rid not in records:
            image=Image.open(path).convert("RGB")
            records[rid]={
                "id":rid,"image":image,"boxes":[],"labels":[],
                "split":str(row.get("split","")).strip().lower(),
            }
        box=[float(row[c]) for c in ("x0","y0","x1","y1")]
        w,h=records[rid]["image"].size
        if not (0<=box[0]<box[2]<=w and 0<=box[1]<box[3]<=h):
            raise ValueError(f"{rid}: invalid box {box}")
        records[rid]["boxes"].append(box)
        records[rid]["labels"].append(str(row["label"]))

    return list(records.values())

if USE_BYOD:
    if not BYOD_ZIP_PATH:
        raise ValueError("Set BYOD_ZIP_PATH for non-interactive BYOD execution")
    byod_records=load_byod(BYOD_ZIP_PATH)
    print({"byod_images":len(byod_records),"classes":sorted({x for r in byod_records for x in r["labels"]})})
else:
    print("BYOD disabled; canonical synthetic E2E path completed.")

## BYOD privacy

User-supplied imagery and annotations are processed inside the selected notebook runtime and are not sent to DIMER workers or APIs.

A hosted notebook is still an external compute environment. Do not upload confidential, biometric, surveillance, personal, security-sensitive, restricted, or proprietary imagery unless authorized.

# 21. Interpretation and limits

### Synthetic domain

These are rendered signs, not photographs.

High AP means that bounded adaptation learned this controlled rendering domain. It does not establish performance on real traffic signs.

### Small test set

Twelve images are sample-sanity evidence, not a benchmark.

### Different model-native recipes

This notebook compares **qualified detector systems**, not architecture under one optimizer.

RT-DETR and YOLOX differ in:

- prediction representation;
- matching/assignment;
- loss;
- optimizer;
- trainable surface;
- post-processing.

### AP50 can hide localization differences

AP50 may saturate. Read AP75 and AP@[.50:.95] alongside it.

### Scores are not calibrated probabilities

RT-DETR's class sigmoid and YOLOX's objectness × class score have different semantics.

### Closed vocabulary

After adaptation, this exercise's detectors recognize only:

- stop-sign
- yield-sign
- speed-limit-sign

They cannot accept a new text label at inference time.

That limitation motivates the subsequent **Open-Vocabulary Detection & Counting** workshop.

# 22. Exercises

### Exercise A — set prediction versus dense detection

Why does RT-DETR not require conventional NMS while YOLOX does?

### Exercise B — AP50 versus AP75

If two detectors both reach AP50 = 1.0, can one still localize boxes more accurately?

### Exercise C — bigger YOLOX

In `FULL`, does the 99M-parameter YOLOX-X outperform the 9M-parameter YOLOX-S on this small task?

Do not assume the answer before running it.

### Exercise D — score threshold

What changes when display threshold moves from 0.05 to 0.70?

Why should test data never be used to choose that operating point?

### Exercise E — production transfer

What labelled evidence would be required before adapting either detector to real Philippine road-sign imagery?

# 23. Export provenance and report

In [ ]:
# @title Final provenance
import datetime

experiment_manifest={
    "notebook_spec":"2.1",
    "notebook_profile":"E2E",
    "notebook_mode":"WORKSHOP",
    "workshop_revision":"0.1.0-candidate",
    "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "execution_tier":WORKSHOP_TIER,
    "dataset":dataset_manifest,
    "models":frozen_experiment["models"],
    "evaluation":frozen_experiment["evaluation"],
    "training_reports":training_reports,
    "test_metrics":test_table.to_dict("records"),
    "artifact_parity":parity,
    "standalone_contract":{
        "dimer_git_clone_required":False,
        "dimer_source_runtime_fetch_required":False,
        "dimer_service_required":False,
        "credentials_required":False,
        "default_upload_required":False,
    },
    "candidate_release_gates":[
        "carry pinned upstream YOLOX modules inline in notebook cells rather than fetch them at runtime",
        "fresh T4 Run all for STANDARD",
        "fresh T4 Run all for FULL",
        "record exact peak VRAM and wall time",
    ],
    "evidence_scope":"60 deterministic rendered sign scenes; 36 train / 12 validation / 12 independent test",
}

(OUTPUT_ROOT/"provenance"/"experiment_manifest.json").write_text(
    json.dumps(experiment_manifest,indent=2,default=str),encoding="utf-8"
)
(OUTPUT_ROOT/"workshop_summary.json").write_text(
    json.dumps({
        "tier":WORKSHOP_TIER,
        "models":MODEL_KEYS,
        "dataset_sha256":dataset_manifest["dataset_sha256"],
        "test_metrics":test_table.to_dict("records"),
        "artifacts":artifact_descriptors,
        "reload_parity":parity,
    },indent=2,default=str),
    encoding="utf-8"
)

import shutil
bundle=shutil.make_archive(
    str(Path(OUTPUT_DIR).resolve())+"_DIMER_Closed_Set_Detection_Workshop_Report",
    "zip",
    root_dir=Path(OUTPUT_DIR).resolve(),
)
print({"bundle":bundle,"sha256":sha256_file(bundle)})

# 24. Troubleshooting

| Symptom | Likely cause | Corrective action |
|---|---|---|
| CUDA unavailable | CPU runtime selected | choose a T4-class runtime before `Run all` |
| RT-DETR digest mismatch | incomplete/changed Hub asset | remove cached snapshot and retry; never bypass digest |
| YOLOX source digest mismatch | pinned upstream file changed/incorrect response | stop; never execute unverified source |
| YOLOX checkpoint digest mismatch | incomplete/changed release asset | remove cached checkpoint and retry |
| YOLOX OOM during SimOTA | label-assignment memory pressure | use canonical T4/batch size; do not change test protocol silently |
| one class absent from a split | canonical/BYOD split invalid | supply more data or explicit split; do not move boxes ad hoc |
| AP50 high but AP low | boxes not tight enough at stricter IoU thresholds | inspect localization, not only class hits |
| training loss decreases but validation AP does not | optimization did not transfer | keep the fixed recipe and interpret the result |
| reload parity fails | artifact/base mismatch or nondeterministic boundary | refuse the artifact and inspect identity/state |
| BYOD geometry differs between models | RT-DETR squash vs YOLOX letterbox | interpret BYOD as system-level rather than architecture-only comparison |

Failure should remain explicit rather than silently changing the detection problem.

# Glossary

| Term | Meaning |
|---|---|
| **Closed-set detector** | Detector with a fixed learned class vocabulary |
| **Bounding box** | Axis-aligned `[x0,y0,x1,y1]` rectangle around an object |
| **IoU** | Intersection-over-union between predicted and reference boxes |
| **AP** | Mean average precision across IoU thresholds 0.50–0.95 |
| **AP50** | AP using IoU ≥ 0.50 |
| **AP75** | AP using IoU ≥ 0.75 |
| **Object query** | Learned RT-DETR decoder query that can predict one object |
| **Objectness** | YOLOX estimate that a candidate location contains an object |
| **SimOTA** | YOLOX dynamic target-assignment procedure |
| **NMS** | Non-maximum suppression used to remove overlapping dense predictions |
| **Set prediction** | DETR-style formulation predicting a bounded object set without conventional NMS |
| **Re-head** | Replace/reinitialize the classification head for a new vocabulary |
| **Sample-sanity evidence** | Bounded tutorial evidence, not a deployment benchmark |

In [ ]:
# @title Run-all completion summary
summary={
    "notebook_spec":"2.1",
    "profile":"E2E",
    "mode":"WORKSHOP",
    "tier":WORKSHOP_TIER,
    "models":MODEL_KEYS,
    "dataset_sha256":dataset_manifest["dataset_sha256"],
    "split_counts":{k:len(v) for k,v in splits.items()},
    "artifacts":artifact_descriptors,
    "reload_parity":parity,
    "output_directory":str(OUTPUT_ROOT.resolve()),
    "release_status":"candidate",
}
display(pd.Series(summary,name="value").to_frame())
print(
    "Closed-set detection notebook complete. "
    "Interpret built-in metrics as synthetic sample-sanity evidence; "
    "release still requires inline YOLOX source carrying plus fresh T4 qualification."
)